# Energy Analytics Platform: Gold pipeline

Spark Declarative Pipelines source file. It is **not run cell by cell**: attach it to a pipeline and
run the pipeline. Pipeline settings: default catalog `dbw_energy_analytics_dev`, default schema `gold`.

Reads the silver current-value tables and creates four materialized views in the `gold` schema:

| view | one row is | replaces |
|---|---|---|
| `gold_hourly_energy_balance` | an hour and area: demand, forecast, net generation, interchange | notebook section 1 |
| `gold_hourly_fuel_mix` | an hour, area and fuel: generation | notebook section 2 |
| `gold_energy_coverage` | an area: how many hours have demand, generation, interchange | validation (first table) |
| `gold_fuel_coverage` | an area and fuel: how many hourly records | validation (second table) |

Run the silver pipeline first: this pipeline reads its tables.

In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "dbw_energy_analytics_dev"

REGION_CURRENT = f"{CATALOG}.silver.region_current"
FUEL_CURRENT = f"{CATALOG}.silver.fuel_current"

In [0]:
# period is text such as 2026-11-07T07 (an hour, in UTC).
# try_to_timestamp gives null instead of failing on bad text, and the trailing Z makes the
# result the same instant whatever the session time zone is.
PERIOD_UTC = F.expr(
    "try_to_timestamp(concat(period, ':00:00Z'), \"yyyy-MM-dd'T'HH:mm:ssX\")"
)


def add_time_columns(df):
    """Adds period_utc, year, month and season (all UTC).

    year and month are read from the period text, not from the timestamp, so the result does not depend
    on the session time zone (a pipeline has no notebook cell to set it in).
    """
    return (
        df
        .withColumn("period_utc", PERIOD_UTC)
        .withColumn("year", F.expr("try_cast(substr(period, 1, 4) AS INT)"))
        .withColumn("month", F.expr("try_cast(substr(period, 6, 2) AS INT)"))
        .withColumn(
            "season",
            F.when(F.col("month").isNull(), F.lit(None).cast("string"))
             .when(F.col("month").isin(12, 1, 2), "Winter")
             .when(F.col("month").isin(3, 4, 5), "Spring")
             .when(F.col("month").isin(6, 7, 8), "Summer")
             .otherwise("Autumn"),
        )
    )

In [0]:
# Warn-only checks: failing rows are kept and counted in the pipeline's Data quality tab.
CHECKS_ENERGY = {
    "period_utc_present": "period_utc IS NOT NULL",
    # net generation = demand + interchange, with 1 MWh of room for rounding
    "balance_ok": "balance_difference_mwh IS NULL OR abs(balance_difference_mwh) <= 1",
}

CHECKS_FUEL = {
    "period_utc_present": "period_utc IS NOT NULL",
    "generation_present": "generation_mwh IS NOT NULL",
}

In [0]:
@dp.materialized_view(
    name=f"{CATALOG}.gold.gold_hourly_energy_balance",
    comment=(
        "One row per hour and area: demand, day-ahead forecast, net generation and total "
        "interchange, with the pull time of each and a check that generation = demand + interchange."
    ),
)
@dp.expect_all(CHECKS_ENERGY)
def gold_hourly_energy_balance():
    region = add_time_columns(spark.read.table(REGION_CURRENT))

    return (
        region
        .groupBy("period", "period_utc", "year", "month", "season", "respondent")
        .agg(
            F.max("respondent_name").alias("respondent_name"),
            F.max(F.when(F.col("type") == "D", F.col("value"))).alias("demand_mwh"),
            F.max(F.when(F.col("type") == "DF", F.col("value"))).alias("forecast_mwh"),
            F.max(F.when(F.col("type") == "NG", F.col("value"))).alias("net_generation_mwh"),
            F.max(F.when(F.col("type") == "TI", F.col("value"))).alias("total_interchange_mwh"),
            F.max(F.when(F.col("type") == "D", F.col("pulled_at"))).alias("demand_pulled_at"),
            F.max(F.when(F.col("type") == "DF", F.col("pulled_at"))).alias("forecast_pulled_at"),
            F.max(F.when(F.col("type") == "NG", F.col("pulled_at"))).alias("net_generation_pulled_at"),
            F.max(F.when(F.col("type") == "TI", F.col("pulled_at"))).alias("interchange_pulled_at"),
            F.max("value_units").alias("value_units"),
        )
        .withColumn(
            "balance_difference_mwh",
            F.when(
                F.col("demand_mwh").isNotNull()
                & F.col("net_generation_mwh").isNotNull()
                & F.col("total_interchange_mwh").isNotNull(),
                F.col("net_generation_mwh")
                - F.col("demand_mwh")
                - F.col("total_interchange_mwh"),
            ),
        )
    )

In [0]:
@dp.materialized_view(
    name=f"{CATALOG}.gold.gold_hourly_fuel_mix",
    comment="One row per hour, area and fuel type: net generation in MWh.",
)
@dp.expect_all(CHECKS_FUEL)
def gold_hourly_fuel_mix():
    fuel = add_time_columns(spark.read.table(FUEL_CURRENT))

    return fuel.select(
        "period",
        "period_utc",
        "respondent",
        "respondent_name",
        "fueltype",
        F.col("type_name").alias("fueltype_name"),
        F.col("value").alias("generation_mwh"),
        "value_units",
        "pulled_at",
        "source_file",
        "year",
        "month",
        "season",
    )

In [0]:
# These two replace the validation cell of the notebook (which used display()).

@dp.materialized_view(
    name=f"{CATALOG}.gold.gold_energy_coverage",
    comment="Per area: hours in the balance table, and how many have demand, generation and interchange.",
)
def gold_energy_coverage():
    return (
        spark.read.table(f"{CATALOG}.gold.gold_hourly_energy_balance")
        .groupBy("respondent")
        .agg(
            F.count("*").alias("hours"),
            F.sum(F.when(F.col("demand_mwh").isNotNull(), 1).otherwise(0)).alias("hours_with_demand"),
            F.sum(F.when(F.col("net_generation_mwh").isNotNull(), 1).otherwise(0)).alias("hours_with_generation"),
            F.sum(F.when(F.col("total_interchange_mwh").isNotNull(), 1).otherwise(0)).alias("hours_with_interchange"),
        )
    )


@dp.materialized_view(
    name=f"{CATALOG}.gold.gold_fuel_coverage",
    comment="Per area and fuel type: number of hourly records.",
)
def gold_fuel_coverage():
    return (
        spark.read.table(f"{CATALOG}.gold.gold_hourly_fuel_mix")
        .groupBy("respondent", "fueltype", "fueltype_name")
        .agg(F.count("*").alias("hourly_records"))
    )

In [0]:
@dp.materialized_view(
    name="gold_peak_demand"
)
def gold_peak_demand():

    energy = spark.read.table(
        "dbw_energy_analytics_dev.gold.gold_hourly_energy_balance"
    )

    window_spec = (
        Window
        .partitionBy("respondent", "year", "season")
        .orderBy(
            F.col("demand_mwh").desc(),
            F.col("period_utc").asc()
        )
    )

    peak = (
        energy
        .filter(F.col("demand_mwh").isNotNull())
        .withColumn(
            "peak_rank",
            F.row_number().over(window_spec)
        )
        .filter(F.col("peak_rank") == 1)
        .drop("peak_rank")
        .withColumn(
            "peak_date_utc",
            F.to_date(F.substring("period", 1, 10))
        )
    )

    return peak

In [0]:
@dp.materialized_view(
    name="gold_peak_demand_fuel_mix"
)
def gold_peak_demand_fuel_mix():

    peak = spark.read.table(
        "dbw_energy_analytics_dev.gold.gold_peak_demand"
    )

    fuel = spark.table(
        "dbw_energy_analytics_dev.gold.gold_hourly_fuel_mix"
    )

    result = (
        peak.alias("p")
        .join(
            fuel.alias("f"),
            (
                (F.col("p.respondent") == F.col("f.respondent")) &
                (F.col("p.period") == F.col("f.period"))
            ),
            "inner"
        )
        .select(
            F.col("p.respondent"),
            F.col("p.year"),
            F.col("p.season"),
            F.col("p.period"),
            F.col("p.period_utc"),
            F.col("p.demand_mwh"),
            F.col("p.forecast_mwh"),
            F.col("p.net_generation_mwh"),
            F.col("p.total_interchange_mwh"),
            F.col("f.fueltype"),
            F.col("f.generation_mwh")
        )
        .withColumn(
            "generation_share_pct",
            F.when(
                F.col("net_generation_mwh") > 0,
                F.round(
                    F.col("generation_mwh") /
                    F.col("net_generation_mwh") * 100,
                    2
                )
            )
        )
    )

    return result

In [0]:
@dp.materialized_view(
    name="gold_forecast_accuracy"
)
def gold_forecast_accuracy():

    energy = spark.read.table(
        "dbw_energy_analytics_dev.gold.gold_hourly_energy_balance"
    )

    valid = (
        energy
        .filter(
            F.col("demand_mwh").isNotNull() &
            F.col("forecast_mwh").isNotNull()
        )
        .withColumn(
            "forecast_error_mwh",
            F.col("forecast_mwh") - F.col("demand_mwh")
        )
        .withColumn(
            "absolute_error_mwh",
            F.abs(F.col("forecast_error_mwh"))
        )
        .withColumn(
            "absolute_percentage_error",
            F.when(
                F.col("demand_mwh") != 0,
                F.abs(F.col("forecast_error_mwh")) /
                F.abs(F.col("demand_mwh")) * 100
            )
        )
    )

    result = (
        valid
        .groupBy(
            "respondent",
            "year",
            "season"
        )
        .agg(
            F.count("*").alias("forecast_observations"),

            F.round(
                F.avg("absolute_error_mwh"), 2
            ).alias("mae_mwh"),

            F.round(
                F.avg("absolute_percentage_error"), 2
            ).alias("mape_pct"),

            F.round(
                F.avg("forecast_error_mwh"), 2
            ).alias("forecast_bias_mwh"),

            F.round(
                F.avg("demand_mwh"), 2
            ).alias("avg_actual_demand_mwh"),

            F.round(
                F.avg("forecast_mwh"), 2
            ).alias("avg_forecast_mwh")
        )
    )

    return result